# RL Trading Agent — Population-Based Training

**Colab Pro runtime required.** Checkpoints save to Google Drive.

- Phase 1: 10 agents, 100 generations (~8 hours on T4)
- Phase 2: 20 agents, 500 generations (~40 hours)

**Setup:** Upload `rl_dataset.npz` to `My Drive/ai-finance/` before running.

In [ ]:
import os, subprocess

# Verify we're on Colab, not local
assert os.path.exists("/content"), "NOT on Colab! Open this notebook in Colab browser."

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpu_info = gpu.stdout.strip() if gpu.returncode == 0 else "No GPU"
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f"Runtime: Colab | GPU: {gpu_info} | RAM: {ram_gb:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/putratogatorop/AI-Finance.git /content/AI-Finance 2>/dev/null || (cd /content/AI-Finance && git pull)
%cd /content/AI-Finance
!git checkout feat/rl-gym-environment
%cd /content/AI-Finance/services/python
!pip install -q torch numpy pandas

In [ ]:
import numpy as np
from pathlib import Path

# Copy dataset from Google Drive (upload rl_dataset.npz to My Drive/ai-finance/ first)
DRIVE_DATA = Path("/content/drive/MyDrive/ai-finance/rl_dataset.npz")
LOCAL_DATA = Path("/content/AI-Finance/data/features/rl_dataset.npz")
LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA.exists(), f"Dataset not found at {DRIVE_DATA}. Upload rl_dataset.npz to My Drive/ai-finance/"
!cp "{DRIVE_DATA}" "{LOCAL_DATA}"

d = np.load(str(LOCAL_DATA), allow_pickle=True)
print(f"Dataset loaded: {d['alt_features'].shape[1]} alts, {d['alt_features'].shape[0]} timestamps, {d['alt_features'].shape[2]} features")

In [ ]:
CKPT = "/content/drive/MyDrive/ai-finance/rl_checkpoints"
!python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 10 --generations 100 --episodes 10

In [ ]:
import json
from pathlib import Path
meta = Path(CKPT) / "metadata.json"
if meta.exists():
    with open(meta) as f: m = json.load(f)
    print(f"Gen: {m['generation']}, Best: {m['best_reward']:.4f}, Mean: {m['mean_reward']:.4f}")

In [ ]:
# !python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 20 --generations 500 --episodes 10

In [ ]:
# !python scripts/evaluate_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT}